# Shadow Price Prediction - Monthly Aggregation Approach

## Overview

This notebook implements a **monthly-aggregated** modeling approach for shadow price prediction:

### Key Differences from `test.ipynb`:

**Original Approach (test.ipynb)**:
- Predict at outage_date level (3-day periods)
- ~3 predictions per branch per month
- Aggregate predictions to monthly level
- More training samples but more noise

**Monthly Aggregation Approach (this notebook)**:
- Aggregate features and labels to monthly level FIRST
- Single prediction per branch per month
- Feature engineering: mean, max, P95, std for each flow percentage
- Fewer training samples but potentially better signal-to-noise ratio

### Monthly Feature Engineering

For each branch_name in each auction_month, we create:
- **Flow statistics** (4 metrics × 5 flow levels = 20 features):
  - Mean flow: Average flow percentage across outage dates
  - Max flow: Peak flow percentage in the month
  - P95 flow: 95th percentile flow
  - Std flow: Flow variability (standard deviation)
- Applied to flow percentages: 100, 90, 80, 70, 60

### Monthly Label Engineering

- **Max shadow price**: Maximum shadow price across all outage dates (primary target for regression)
- **Binding count**: Number of outage dates where constraint bound
- **Binary label**: Whether max_shadow_price > 0 (classification target)

### Expected Performance Trade-offs

**Potential Improvements**:
- Better precision/recall on monthly predictions
- More stable patterns, less overfitting
- Captures both central tendency and variability

**Potential Drawbacks**:
- 1/3 fewer training samples
- Can't capture intra-month dynamics
- May miss important outage-specific patterns

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    mean_absolute_error, mean_squared_error, r2_score,
    confusion_matrix, classification_report
)

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# ============================================================================
# STEP 1: Configuration and Date Ranges
# ============================================================================

print("=" * 80)
print("SHADOW PRICE PREDICTION: MONTHLY AGGREGATION APPROACH")
print("=" * 80)

# Test period: 2025-11
test_auction_month = pd.Timestamp('2025-10')
test_market_month = pd.Timestamp('2025-10')

# Training period: 12 months before test month
train_end = test_auction_month - pd.offsets.MonthBegin(1)
train_start = train_end - pd.offsets.MonthBegin(12)

# Fixed parameters
period_type = 'f0'
class_type = 'onpeak'
market_round = 1

print(f"\n[STEP 1] Configuration")
print(f"  Training Period: {train_start.strftime('%Y-%m')} to {train_end.strftime('%Y-%m')}")
print(f"  Test Period: {test_auction_month.strftime('%Y-%m')}")
print(f"  Period Type: {period_type}")
print(f"  Class Type: {class_type}")
print(f"  Market Round: {market_round}")

# Feature engineering configuration
FLOW_PERCENTILES = ['100', '90', '80', '70', '60']
AGGREGATION_STATS = ['mean', 'max', 'p95', 'std']  # 4 stats per flow level

print(f"\n  Flow Percentiles: {FLOW_PERCENTILES}")
print(f"  Aggregation Stats: {AGGREGATION_STATS}")
print(f"  Total Features: {len(FLOW_PERCENTILES) * len(AGGREGATION_STATS)} (flow stats)")

SHADOW PRICE PREDICTION: MONTHLY AGGREGATION APPROACH

[STEP 1] Configuration
  Training Period: 2024-09 to 2025-09
  Test Period: 2025-10
  Period Type: f0
  Class Type: onpeak
  Market Round: 1

  Flow Percentiles: ['100', '90', '80', '70', '60']
  Aggregation Stats: ['mean', 'max', 'p95', 'std']
  Total Features: 20 (flow stats)


In [4]:
# ============================================================================
# STEP 2: Helper Function to Load Outage-Level Data
# ============================================================================

from pbase.analysis.tools.all_positions import MisoApTools

aptools = MisoApTools()

def load_outage_data(auction_month, market_month, outage_date, 
                     period_type='f0', class_type='onpeak', market_round=1):
    """
    Load data for a single outage date.
    
    Returns:
    --------
    DataFrame with columns: 100, 90, 80, 70, 60, label, branch_name
    Index: (constraint_id, flow_direction)
    """
    try:
        # Build paths
        score_path_str = f'/opt/temp/tmp/pw_data/spice6/prod_f0p_model_miso/density/auction_month={{auction_month}}/market_month={{market_month}}/market_round={market_round}/outage_date={{outage_date}}'
        cons_path_str = f'/opt/temp/tmp/pw_data/spice6/prod_f0p_model_miso/constraint_info/auction_month={{auction_month}}/market_round={market_round}/period_type={{period_type}}/class_type={{class_type}}'
        
        score_path = Path(score_path_str.format(
            auction_month=auction_month.strftime('%Y-%m'),
            market_month=market_month.strftime('%Y-%m'),
            outage_date=outage_date.strftime('%Y-%m-%d')
        ))
        
        cons_path = Path(cons_path_str.format(
            auction_month=auction_month.strftime('%Y-%m'),
            period_type=period_type,
            class_type=class_type
        ))
        
        # Check if paths exist
        if not score_path.exists():
            return None
        
        # Load score data (flow percentages)
        score_df = pd.read_parquet(score_path / 'score_df.parquet')
        
        # Transform cumulative to incremental probabilities
        score_df['60'] -= score_df['70']
        score_df['70'] -= score_df['80']
        score_df['80'] -= score_df['90']
        score_df['90'] -= score_df['100']
        
        # Load shadow prices for the outage period using aptools
        outage_st = outage_date
        outage_et = outage_date + pd.Timedelta(days=4)
        
        da_label = aptools.tools.get_da_shadow_by_peaktype(
            market_month, 
            market_month + pd.offsets.MonthBegin(1), 
            class_type
        )
        da_label.index = da_label.index.tz_localize(None)
        
        # Aggregate shadow prices for this outage period (SUM over the period)
        da_label_chunk = da_label.loc[
            (da_label.index >= outage_st) & (da_label.index < outage_et)
        ].groupby('constraint_id')['shadow_price'].sum()
        
        # Load constraint info to map constraint_id to branch_name
        if cons_path.exists():
            cons = pd.read_parquet(cons_path / 'constraints.parquet')
            # Create spice_map: constraint_id -> branch_name
            spice_map = cons[cons['convention'] != 999].dropna(subset=['branch_name']).groupby('constraint_id')['branch_name'].apply(','.join)
            
            # Map constraint_id to branch_name in both DataFrames
            score_df['branch_name'] = score_df.index.get_level_values('constraint_id').map(spice_map)
            da_label_chunk.index = da_label_chunk.index.map(spice_map)
            
            # Group by branch_name and sum shadow prices
            da_label_chunk = da_label_chunk.groupby(level=0).sum().abs()
        else:
            # Fallback: use constraint_id as branch_name
            score_df['branch_name'] = score_df.index.get_level_values('constraint_id')
            da_label_chunk = da_label_chunk.abs()
        
        # Assign labels to score_df
        score_df['label'] = score_df['branch_name'].map(da_label_chunk).fillna(0)
        
        # Keep only one row per constraint_id (highest score)
        score_df = score_df.sort_values(
            ['100', '90', '80', '70', '60'], 
            ascending=False
        ).reset_index().groupby('constraint_id').first().reset_index().set_index(
            ['constraint_id', 'flow_direction']
        )
        
        # Select relevant columns
        result = score_df[['100', '90', '80', '70', '60', 'label', 'branch_name']].copy()
        
        return result
        
    except Exception as e:
        print(f"  ⚠️  Error loading {outage_date.strftime('%Y-%m-%d')}: {e}")
        return None

print("✓ Helper function defined: load_outage_data()")
print("  Uses aptools.tools.get_da_shadow_by_peaktype() for correct shadow price loading")

✓ Helper function defined: load_outage_data()
  Uses aptools.tools.get_da_shadow_by_peaktype() for correct shadow price loading


In [5]:
# ============================================================================
# STEP 3: Monthly Aggregation Function
# ============================================================================

def aggregate_to_monthly(monthly_data_list, auction_month_label):
    """
    Aggregate outage-level data to monthly level per branch_name.
    
    Parameters:
    -----------
    monthly_data_list : list of DataFrames
        Each DataFrame contains outage-level data for one outage_date
    auction_month_label : Timestamp
        The auction month for labeling
    
    Returns:
    --------
    DataFrame with monthly aggregated features and labels per branch_name
    Columns: auction_month, branch_name, [flow features], max_label, binding_count
    """
    if not monthly_data_list:
        return None
    
    # Concatenate all outage data for this month
    all_data = pd.concat(monthly_data_list, ignore_index=False)
    
    # Add branch_name to index for grouping
    all_data_reset = all_data.reset_index()
    
    # Group by branch_name and aggregate
    agg_dict = {}
    
    # Flow percentiles: compute mean, max, p95, std
    for flow_pct in ['100', '90', '80', '70', '60']:
        agg_dict[f'flow_{flow_pct}_mean'] = (flow_pct, 'mean')
        agg_dict[f'flow_{flow_pct}_max'] = (flow_pct, 'max')
        agg_dict[f'flow_{flow_pct}_p95'] = (flow_pct, lambda x: np.percentile(x, 95))
        agg_dict[f'flow_{flow_pct}_std'] = (flow_pct, 'std')
    
    # Labels: max shadow price and binding count
    agg_dict['max_label'] = ('label', 'max')
    agg_dict['mean_label'] = ('label', 'mean')
    agg_dict['binding_count'] = ('label', lambda x: (x > 0).sum())
    
    # Perform aggregation
    monthly_agg = all_data_reset.groupby('branch_name').agg(**agg_dict).reset_index()
    
    # Fill NaN in std with 0 (for branches with single observation)
    for flow_pct in ['100', '90', '80', '70', '60']:
        monthly_agg[f'flow_{flow_pct}_std'] = monthly_agg[f'flow_{flow_pct}_std'].fillna(0)
    
    # Add auction_month column
    monthly_agg['auction_month'] = auction_month_label
    
    return monthly_agg

print("✓ Helper function defined: aggregate_to_monthly()")
print("  Creates 20 flow features (4 stats × 5 percentiles) + max_label + binding_count")

✓ Helper function defined: aggregate_to_monthly()
  Creates 20 flow features (4 stats × 5 percentiles) + max_label + binding_count


In [6]:
# ============================================================================
# STEP 4: Load and Aggregate Training Data (Monthly)
# ============================================================================

print("\n[STEP 4] Loading and Aggregating Training Data to Monthly Level")
print("-" * 80)

# Generate list of training months
train_months = pd.date_range(start=train_start, end=train_end, freq='MS')
print(f"  Training months: {len(train_months)}")

# Storage for monthly aggregated data
monthly_train_data = []

for auction_month in train_months:
    market_month = auction_month
    
    # Get all outage dates for this month (typically 3 per month, every ~10 days)
    month_start = auction_month
    month_end = auction_month + pd.offsets.MonthEnd(1)
    
    # Generate outage dates: start from 1st, 11th, 21st of each month
    outage_dates = []
    for day in [1, 11, 21]:
        outage_date = auction_month.replace(day=day)
        if outage_date <= month_end:
            outage_dates.append(outage_date)
    
    print(f"\n  Processing {auction_month.strftime('%Y-%m')}: {len(outage_dates)} outage dates")
    
    # Load data for all outage dates in this month
    month_outage_data = []
    
    for outage_date in outage_dates:
        outage_data = load_outage_data(
            auction_month=auction_month,
            market_month=market_month,
            outage_date=outage_date,
            period_type=period_type,
            class_type=class_type,
            market_round=market_round
        )
        
        if outage_data is not None:
            month_outage_data.append(outage_data)
            print(f"    ✓ {outage_date.strftime('%Y-%m-%d')}: {len(outage_data):,} constraints")
    
    # Aggregate this month's data
    if month_outage_data:
        monthly_agg = aggregate_to_monthly(month_outage_data, auction_month)
        monthly_train_data.append(monthly_agg)
        print(f"    → Monthly aggregation: {len(monthly_agg):,} unique branches")
    else:
        print(f"    ⚠️  No data available for {auction_month.strftime('%Y-%m')}")

# Combine all monthly training data
if monthly_train_data:
    train_data_monthly = pd.concat(monthly_train_data, ignore_index=True)
    print(f"\n✓ Training Data Loaded and Aggregated")
    print(f"  Total monthly samples: {len(train_data_monthly):,}")
    print(f"  Unique branches: {train_data_monthly['branch_name'].nunique():,}")
    print(f"  Months with data: {train_data_monthly['auction_month'].nunique():,}")
    print(f"  Binding samples (max_label > 0): {(train_data_monthly['max_label'] > 0).sum():,} "
          f"({(train_data_monthly['max_label'] > 0).mean() * 100:.1f}%)")
else:
    train_data_monthly = None
    print("\n⚠️  ERROR: No training data loaded")


[STEP 4] Loading and Aggregating Training Data to Monthly Level
--------------------------------------------------------------------------------
  Training months: 13

  Processing 2024-09: 3 outage dates
    ✓ 2024-09-01: 12,636 constraints
    → Monthly aggregation: 4,175 unique branches

  Processing 2024-10: 3 outage dates
    ✓ 2024-10-01: 12,177 constraints
    → Monthly aggregation: 4,022 unique branches

  Processing 2024-11: 3 outage dates
    ✓ 2024-11-01: 12,335 constraints
    → Monthly aggregation: 4,062 unique branches

  Processing 2024-12: 3 outage dates
    ✓ 2024-12-01: 12,342 constraints
    → Monthly aggregation: 4,052 unique branches

  Processing 2025-01: 3 outage dates
    ✓ 2025-01-01: 12,720 constraints
    → Monthly aggregation: 4,178 unique branches

  Processing 2025-02: 3 outage dates
    ✓ 2025-02-01: 12,399 constraints
    → Monthly aggregation: 4,081 unique branches

  Processing 2025-03: 3 outage dates
    ✓ 2025-03-01: 12,413 constraints
    → Monthly

In [7]:
# ============================================================================
# STEP 5: Load and Aggregate Test Data (Monthly)
# ============================================================================

print("\n[STEP 5] Loading and Aggregating Test Data to Monthly Level")
print("-" * 80)

# Get outage dates for test month: 1st, 11th, 21st
test_month_end = test_auction_month + pd.offsets.MonthEnd(1)

test_outage_dates = []
for day in [1, 11, 21]:
    outage_date = test_auction_month.replace(day=day)
    if outage_date <= test_month_end:
        test_outage_dates.append(outage_date)

print(f"  Test month: {test_auction_month.strftime('%Y-%m')}")
print(f"  Outage dates: {len(test_outage_dates)}")

# Load data for all test outage dates
test_outage_data = []

for outage_date in test_outage_dates:
    outage_data = load_outage_data(
        auction_month=test_auction_month,
        market_month=test_market_month,
        outage_date=outage_date,
        period_type=period_type,
        class_type=class_type,
        market_round=market_round
    )
    
    if outage_data is not None:
        test_outage_data.append(outage_data)
        print(f"  ✓ {outage_date.strftime('%Y-%m-%d')}: {len(outage_data):,} constraints")

# Aggregate test data to monthly
if test_outage_data:
    test_data_monthly = aggregate_to_monthly(test_outage_data, test_auction_month)
    print(f"\n✓ Test Data Loaded and Aggregated")
    print(f"  Monthly test samples: {len(test_data_monthly):,}")
    print(f"  Unique branches: {test_data_monthly['branch_name'].nunique():,}")
    print(f"  Binding samples (max_label > 0): {(test_data_monthly['max_label'] > 0).sum():,} "
          f"({(test_data_monthly['max_label'] > 0).mean() * 100:.1f}%)")
else:
    test_data_monthly = None
    print("\n⚠️  ERROR: No test data loaded")


[STEP 5] Loading and Aggregating Test Data to Monthly Level
--------------------------------------------------------------------------------
  Test month: 2025-10
  Outage dates: 3
  ✓ 2025-10-01: 12,646 constraints

✓ Test Data Loaded and Aggregated
  Monthly test samples: 4,082
  Unique branches: 4,082
  Binding samples (max_label > 0): 76 (1.9%)


In [8]:
# ============================================================================
# STEP 6: Prepare Features and Targets
# ============================================================================

print("\n[STEP 6] Preparing Features and Targets")
print("-" * 80)

if train_data_monthly is not None and test_data_monthly is not None:
    
    # Define feature columns (20 flow features)
    feature_cols = []
    for flow_pct in ['100', '90', '80', '70', '60']:
        for stat in ['mean', 'max', 'p95', 'std']:
            feature_cols.append(f'flow_{flow_pct}_{stat}')
    
    print(f"  Feature columns: {len(feature_cols)}")
    print(f"  Example features: {feature_cols[:4]} ... {feature_cols[-4:]}")
    
    # Separate by branch_name for branch-specific models
    train_data_by_branch = {}
    for branch_name, group in train_data_monthly.groupby('branch_name'):
        train_data_by_branch[branch_name] = group
    
    print(f"\n  Training data by branch:")
    print(f"    Total branches in training: {len(train_data_by_branch):,}")
    print(f"    Avg samples per branch: {len(train_data_monthly) / len(train_data_by_branch):.1f}")
    
    # Check overlap with test branches
    test_branches = set(test_data_monthly['branch_name'].unique())
    train_branches = set(train_data_by_branch.keys())
    overlap = test_branches & train_branches
    
    print(f"\n  Branch overlap analysis:")
    print(f"    Test branches: {len(test_branches):,}")
    print(f"    Training branches: {len(train_branches):,}")
    print(f"    Overlap: {len(overlap):,} ({len(overlap)/len(test_branches)*100:.1f}% of test)")
    print(f"    Unseen in test: {len(test_branches - train_branches):,}")
    
else:
    print("⚠️  ERROR: Cannot prepare features - missing training or test data")


[STEP 6] Preparing Features and Targets
--------------------------------------------------------------------------------
  Feature columns: 20
  Example features: ['flow_100_mean', 'flow_100_max', 'flow_100_p95', 'flow_100_std'] ... ['flow_60_mean', 'flow_60_max', 'flow_60_p95', 'flow_60_std']

  Training data by branch:
    Total branches in training: 4,482
    Avg samples per branch: 11.9

  Branch overlap analysis:
    Test branches: 4,082
    Training branches: 4,482
    Overlap: 4,063 (99.5% of test)
    Unseen in test: 19


In [9]:
# ============================================================================
# STEP 7: Train Branch-Specific Classification Models
# ============================================================================

print("\n[STEP 7] Training Branch-Specific Classification Models")
print("-" * 80)

from sklearn.metrics import f1_score

def find_optimal_threshold(y_true, y_proba, thresholds=None):
    """
    Find optimal classification threshold that maximizes F1-score.
    """
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 99)
    
    f1_scores = []
    for threshold in thresholds:
        y_pred = (y_proba >= threshold).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        f1_scores.append(f1)
    
    optimal_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[optimal_idx]
    max_f1 = f1_scores[optimal_idx]
    
    return optimal_threshold, max_f1

if train_data_monthly is not None:
    
    # Storage for models and thresholds
    clf_models = {}
    optimal_thresholds = {}
    
    # Train default classifier on all data
    print("Training default fallback classifier...")
    
    X_train_all = train_data_monthly[feature_cols].copy()
    y_train_all = train_data_monthly['max_label'].copy()
    y_train_binary_all = (y_train_all > 0).astype(int)
    
    # Calculate scale_pos_weight
    n_samples = len(y_train_binary_all)
    n_binding = y_train_binary_all.sum()
    n_non_binding = n_samples - n_binding
    
    weight_binding = n_samples / (2 * n_binding) if n_binding > 0 else 1.0
    weight_non_binding = n_samples / (2 * n_non_binding)
    scale_pos_weight_all = weight_binding / weight_non_binding if weight_non_binding > 0 else 1.0
    
    clf_default = XGBClassifier(
        n_estimators=100,
        max_depth=2,
        learning_rate=0.1,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight_all,
        random_state=42,
        n_jobs=-1,
        verbosity=0,
        eval_metric='logloss'
    )
    clf_default.fit(X_train_all, y_train_binary_all)
    
    # Optimize threshold for default
    y_proba_all = clf_default.predict_proba(X_train_all)[:, 1]
    optimal_threshold_default, max_f1_default = find_optimal_threshold(
        y_train_binary_all, y_proba_all
    )
    
    print(f"  ✓ Default classifier trained on {len(X_train_all):,} samples")
    print(f"  ✓ Optimal threshold: {optimal_threshold_default:.3f} (F1={max_f1_default:.3f})")
    
    # Train branch-specific models
    print(f"\nTraining branch-specific classifiers...")
    print(f"  Branches to train: {len(train_data_by_branch):,}")
    
    trained_count = 0
    skipped_count = 0
    min_samples = 5  # Minimum monthly samples needed
    
    for branch_name, branch_data in train_data_by_branch.items():
        # Only train if branch appears in test set
        if branch_name not in test_branches:
            skipped_count += 1
            continue
        
        # Skip if too few samples
        if len(branch_data) < min_samples:
            skipped_count += 1
            continue
        
        X_branch = branch_data[feature_cols].copy()
        y_branch = branch_data['max_label'].copy()
        y_branch_binary = (y_branch > 0).astype(int)
        
        # Calculate branch-specific class weights
        n_branch = len(y_branch_binary)
        n_binding_branch = y_branch_binary.sum()
        n_non_binding_branch = n_branch - n_binding_branch
        
        if n_binding_branch > 0 and n_non_binding_branch > 0:
            weight_binding_b = n_branch / (2 * n_binding_branch)
            weight_non_binding_b = n_branch / (2 * n_non_binding_branch)
            scale_pos_weight_b = weight_binding_b / weight_non_binding_b
        else:
            scale_pos_weight_b = 1.0
        
        # Train classifier
        clf_branch = XGBClassifier(
            n_estimators=50,
            max_depth=2,
            learning_rate=0.1,
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight_b,
            random_state=42,
            n_jobs=1,
            verbosity=0,
            eval_metric='logloss'
        )
        
        try:
            clf_branch.fit(X_branch, y_branch_binary)
            clf_models[branch_name] = clf_branch
            
            # Optimize threshold
            y_proba_branch = clf_branch.predict_proba(X_branch)[:, 1]
            optimal_threshold_b, max_f1_b = find_optimal_threshold(
                y_branch_binary, y_proba_branch
            )
            optimal_thresholds[branch_name] = optimal_threshold_b
            
            trained_count += 1
        except Exception as e:
            skipped_count += 1
            continue
    
    print(f"\n✓ Classification Training Complete")
    print(f"  Models trained: {trained_count:,}")
    print(f"  Thresholds optimized: {len(optimal_thresholds):,}")
    print(f"  Branches skipped: {skipped_count:,}")
    
else:
    print("⚠️  ERROR: No training data available")


[STEP 7] Training Branch-Specific Classification Models
--------------------------------------------------------------------------------
Training default fallback classifier...
  ✓ Default classifier trained on 53,432 samples
  ✓ Optimal threshold: 0.870 (F1=0.200)

Training branch-specific classifiers...
  Branches to train: 4,482

✓ Classification Training Complete
  Models trained: 404
  Thresholds optimized: 404
  Branches skipped: 4,078


In [10]:
# ============================================================================
# STEP 8: Train Branch-Specific Regression Models
# ============================================================================

print("\n[STEP 8] Training Branch-Specific Regression Models")
print("-" * 80)

if train_data_monthly is not None:
    
    # Storage for regression models
    reg_models = {}
    
    # Train default regressor on all binding data
    print("Training default fallback regressor...")
    
    binding_mask_all = y_train_all > 0
    X_train_binding_all = X_train_all[binding_mask_all]
    y_train_binding_all = y_train_all[binding_mask_all]
    
    if len(X_train_binding_all) > 5:
        reg_default = XGBRegressor(
            n_estimators=100,
            max_depth=2,
            learning_rate=0.1,
            min_child_weight=2,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            verbosity=0,
            objective='reg:squarederror'
        )
        reg_default.fit(X_train_binding_all, y_train_binding_all)
        print(f"  ✓ Default regressor trained on {len(X_train_binding_all):,} binding samples")
    else:
        reg_default = None
        print(f"  ⚠️  Insufficient binding samples for default regressor")
    
    # Train branch-specific regressors
    print(f"\nTraining branch-specific regressors...")
    
    trained_reg_count = 0
    skipped_reg_count = 0
    min_binding_samples = 1
    
    for branch_name, branch_data in train_data_by_branch.items():
        if branch_name not in test_branches:
            skipped_reg_count += 1
            continue
        
        X_branch = branch_data[feature_cols].copy()
        y_branch = branch_data['max_label'].copy()
        
        # Extract binding samples
        binding_mask = y_branch > 0
        X_branch_binding = X_branch[binding_mask]
        y_branch_binding = y_branch[binding_mask]
        
        if len(X_branch_binding) < min_binding_samples:
            skipped_reg_count += 1
            continue
        
        # Train regressor
        reg_branch = XGBRegressor(
            n_estimators=50,
            max_depth=2,
            learning_rate=0.1,
            min_child_weight=1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=1,
            verbosity=0,
            objective='reg:squarederror'
        )
        
        try:
            reg_branch.fit(X_branch_binding, y_branch_binding)
            reg_models[branch_name] = reg_branch
            trained_reg_count += 1
        except Exception as e:
            skipped_reg_count += 1
            continue
    
    print(f"\n✓ Regression Training Complete")
    print(f"  Regression models trained: {trained_reg_count:,}")
    print(f"  Branches skipped: {skipped_reg_count:,}")
    print(f"  Default regressor: {'Available' if reg_default is not None else 'Not available'}")
    
else:
    print("⚠️  ERROR: No training data available")


[STEP 8] Training Branch-Specific Regression Models
--------------------------------------------------------------------------------
Training default fallback regressor...
  ✓ Default regressor trained on 830 binding samples

Training branch-specific regressors...

✓ Regression Training Complete
  Regression models trained: 408
  Branches skipped: 4,074
  Default regressor: Available


In [11]:
# ============================================================================
# STEP 9: Make Predictions on Test Data
# ============================================================================

print("\n[STEP 9] Making Predictions on Test Data")
print("-" * 80)

if test_data_monthly is not None:
    
    # Initialize prediction arrays
    n_test = len(test_data_monthly)
    y_pred_binary = np.zeros(n_test, dtype=int)
    y_pred_proba = np.zeros(n_test, dtype=float)
    y_pred_shadow_price = np.zeros(n_test, dtype=float)
    
    # Get test features and true labels
    X_test = test_data_monthly[feature_cols].copy()
    y_test = test_data_monthly['max_label'].copy()
    y_test_binary = (y_test > 0).astype(int)
    test_branch_names = test_data_monthly['branch_name'].values
    
    # Track prediction strategies
    branch_specific_count = 0
    default_count = 0
    
    # Predict for each test sample
    for i in range(n_test):
        branch_name = test_branch_names[i]
        X_sample = X_test.iloc[i:i+1]
        
        # Classification: binding or not
        if branch_name in clf_models:
            clf = clf_models[branch_name]
            threshold = optimal_thresholds.get(branch_name, optimal_threshold_default)
            branch_specific_count += 1
        else:
            clf = clf_default
            threshold = optimal_threshold_default
            default_count += 1
        
        y_proba_sample = clf.predict_proba(X_sample)[0, 1]
        y_pred_binary[i] = 1 if y_proba_sample >= threshold else 0
        y_pred_proba[i] = y_proba_sample
        
        # Regression: shadow price magnitude (only if predicted binding)
        if y_pred_binary[i] == 1:
            if branch_name in reg_models:
                reg = reg_models[branch_name]
            else:
                reg = reg_default
            
            if reg is not None:
                y_pred_shadow_price[i] = reg.predict(X_sample)[0]
            else:
                y_pred_shadow_price[i] = 0.0
        else:
            y_pred_shadow_price[i] = 0.0
    
    print(f"\n✓ Predictions Complete")
    print(f"  Total test samples: {n_test:,}")
    print(f"  Branch-specific models used: {branch_specific_count:,} ({branch_specific_count/n_test*100:.1f}%)")
    print(f"  Default model used: {default_count:,} ({default_count/n_test*100:.1f}%)")
    print(f"  Predicted binding: {y_pred_binary.sum():,} ({y_pred_binary.mean()*100:.1f}%)")
    print(f"  Actual binding: {y_test_binary.sum():,} ({y_test_binary.mean()*100:.1f}%)")
    
else:
    print("⚠️  ERROR: No test data available")


[STEP 9] Making Predictions on Test Data
--------------------------------------------------------------------------------

✓ Predictions Complete
  Total test samples: 4,082
  Branch-specific models used: 404 (9.9%)
  Default model used: 3,678 (90.1%)
  Predicted binding: 481 (11.8%)
  Actual binding: 76 (1.9%)


In [12]:
# ============================================================================
# STEP 10: Evaluate Classification Performance
# ============================================================================

print("\n[STEP 10] Classification Performance Evaluation")
print("=" * 80)

if test_data_monthly is not None:
    
    # Calculate classification metrics
    precision = precision_score(y_test_binary, y_pred_binary, zero_division=0)
    recall = recall_score(y_test_binary, y_pred_binary, zero_division=0)
    f1 = f1_score(y_test_binary, y_pred_binary, zero_division=0)
    
    print("\nClassification Metrics (Monthly Level):")
    print("-" * 80)
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    
    # Confusion matrix
    cm = confusion_matrix(y_test_binary, y_pred_binary)
    
    print("\nConfusion Matrix:")
    print("-" * 80)
    print(f"  True Negatives:  {cm[0, 0]:,}")
    print(f"  False Positives: {cm[0, 1]:,}")
    print(f"  False Negatives: {cm[1, 0]:,}")
    print(f"  True Positives:  {cm[1, 1]:,}")
    
    # Classification report
    print("\nDetailed Classification Report:")
    print("-" * 80)
    print(classification_report(y_test_binary, y_pred_binary, 
                                 target_names=['Non-Binding', 'Binding'],
                                 zero_division=0))
    
else:
    print("⚠️  ERROR: No predictions available")


[STEP 10] Classification Performance Evaluation

Classification Metrics (Monthly Level):
--------------------------------------------------------------------------------
  Precision: 0.0956
  Recall:    0.6053
  F1-Score:  0.1652

Confusion Matrix:
--------------------------------------------------------------------------------
  True Negatives:  3,571
  False Positives: 435
  False Negatives: 30
  True Positives:  46

Detailed Classification Report:
--------------------------------------------------------------------------------
              precision    recall  f1-score   support

 Non-Binding       0.99      0.89      0.94      4006
     Binding       0.10      0.61      0.17        76

    accuracy                           0.89      4082
   macro avg       0.54      0.75      0.55      4082
weighted avg       0.97      0.89      0.92      4082



In [13]:
# ============================================================================
# STEP 11: Evaluate Regression Performance
# ============================================================================

print("\n[STEP 11] Regression Performance Evaluation")
print("=" * 80)

if test_data_monthly is not None:
    
    # Evaluate on binding samples only
    binding_mask_test = y_test_binary == 1
    
    if binding_mask_test.sum() > 0:
        y_true_binding = y_test[binding_mask_test]
        y_pred_binding = y_pred_shadow_price[binding_mask_test]
        
        # Calculate regression metrics
        mae = mean_absolute_error(y_true_binding, y_pred_binding)
        rmse = np.sqrt(mean_squared_error(y_true_binding, y_pred_binding))
        r2 = r2_score(y_true_binding, y_pred_binding)
        
        print("\nRegression Metrics (Binding Samples Only):")
        print("-" * 80)
        print(f"  MAE (Mean Absolute Error): ${mae:,.2f}")
        print(f"  RMSE (Root Mean Squared Error): ${rmse:,.2f}")
        print(f"  R² Score: {r2:.4f}")
        
        # Summary statistics
        print("\nShadow Price Statistics (Binding Samples):")
        print("-" * 80)
        print(f"  True - Mean:   ${y_true_binding.mean():,.2f}")
        print(f"  True - Median: ${y_true_binding.median():,.2f}")
        print(f"  True - Max:    ${y_true_binding.max():,.2f}")
        print(f"  True - Min:    ${y_true_binding.min():,.2f}")
        print("")
        print(f"  Pred - Mean:   ${y_pred_binding.mean():,.2f}")
        print(f"  Pred - Median: ${np.median(y_pred_binding):,.2f}")
        print(f"  Pred - Max:    ${y_pred_binding.max():,.2f}")
        print(f"  Pred - Min:    ${y_pred_binding.min():,.2f}")
        
    else:
        print("\n⚠️  No binding samples in test data for regression evaluation")
    
    # Overall regression metrics (all samples)
    mae_all = mean_absolute_error(y_test, y_pred_shadow_price)
    rmse_all = np.sqrt(mean_squared_error(y_test, y_pred_shadow_price))
    
    print("\n\nRegression Metrics (All Samples):")
    print("-" * 80)
    print(f"  MAE:  ${mae_all:,.2f}")
    print(f"  RMSE: ${rmse_all:,.2f}")
    
else:
    print("⚠️  ERROR: No predictions available")


[STEP 11] Regression Performance Evaluation

Regression Metrics (Binding Samples Only):
--------------------------------------------------------------------------------
  MAE (Mean Absolute Error): $1,007.45
  RMSE (Root Mean Squared Error): $1,895.48
  R² Score: -0.0346

Shadow Price Statistics (Binding Samples):
--------------------------------------------------------------------------------
  True - Mean:   $1,228.57
  True - Median: $410.15
  True - Max:    $10,090.04
  True - Min:    $0.01

  Pred - Mean:   $464.63
  Pred - Median: $38.36
  Pred - Max:    $7,944.73
  Pred - Min:    $0.00


Regression Metrics (All Samples):
--------------------------------------------------------------------------------
  MAE:  $77.01
  RMSE: $451.86


In [ ]:
# ============================================================================
# STEP 12: Summary and Next Steps
# ============================================================================

print("\n" + "=" * 80)
print("MONTHLY AGGREGATION APPROACH - SUMMARY")
print("=" * 80)

print("\n📊 Key Findings:")
print("-" * 80)

if test_data_monthly is not None:
    print(f"\n1. Classification Performance (Monthly Level):")
    print(f"   • Precision: {precision:.4f}")
    print(f"   • Recall:    {recall:.4f}")
    print(f"   • F1-Score:  {f1:.4f}")
    
    if binding_mask_test.sum() > 0:
        print(f"\n2. Regression Performance (Binding Samples):")
        print(f"   • MAE:  ${mae:,.2f}")
        print(f"   • RMSE: ${rmse:,.2f}")
        print(f"   • R²:   {r2:.4f}")
    
    print(f"\n3. Data Characteristics:")
    print(f"   • Training samples:  {len(train_data_monthly):,} (monthly aggregated)")
    print(f"   • Test samples:      {len(test_data_monthly):,} (monthly aggregated)")
    print(f"   • Feature count:     {len(feature_cols)} (4 stats × 5 flow percentiles)")
    print(f"   • Models trained:    {len(clf_models):,} classifiers, {len(reg_models):,} regressors")
    
    print(f"\n4. Class Balance:")
    print(f"   • Training binding rate: {(train_data_monthly['max_label'] > 0).mean()*100:.2f}%")
    print(f"   • Test binding rate:     {y_test_binary.mean()*100:.2f}%")
    print(f"   • Predicted binding rate: {y_pred_binary.mean()*100:.2f}%")

print("\n\n💡 Comparison with Outage-Level Approach:")
print("-" * 80)
print("\nTo compare this monthly aggregation approach with the outage-level approach:")
print("1. Run the original test.ipynb for the same test period")
print("2. Compare:")
print("   • Precision/Recall/F1 scores")
print("   • MAE/RMSE on shadow price predictions")
print("   • Training time (monthly should be ~3x faster)")
print("   • Model complexity and interpretability")

print("\n✅ Advantages of Monthly Approach:")
print("   • Smoother predictions (less noise from daily variations)")
print("   • Richer features (mean, max, p95, std capture patterns)")
print("   • Faster training (fewer samples)")
print("   • More stable monthly forecasts")

print("\n⚠️  Potential Disadvantages:")
print("   • Fewer training samples (1/3 of outage-level)")
print("   • Cannot capture intra-month dynamics")
print("   • May miss important outage-specific patterns")
print("   • Higher variance with limited historical data")

print("\n" + "=" * 80)

In [ ]:
# ============================================================================
# BONUS: Feature Importance Analysis
# ============================================================================

print("\n[BONUS] Feature Importance Analysis")
print("=" * 80)

# Get feature importance from default classifier
if clf_default is not None:
    feature_importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': clf_default.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\nTop 10 Most Important Features (Classification):")
    print("-" * 80)
    print(feature_importance.head(10).to_string(index=False))
    
    print("\n\nInterpretation Guide:")
    print("-" * 80)
    print("Feature naming: flow_{percentile}_{stat}")
    print("  • flow_100_max: Maximum flow at 100% threshold")
    print("  • flow_100_p95: 95th percentile flow at 100% threshold")
    print("  • flow_100_mean: Average flow at 100% threshold")
    print("  • flow_100_std: Flow variability at 100% threshold")
    print("\nKey insights:")
    print("  • Higher percentiles (100, 90) typically more important")
    print("  • Max and P95 often capture extreme flow conditions")
    print("  • Std indicates flow instability (higher variance = higher risk)")